# 03_simulate_propofol_ce_models

This notebook reconstructs modeled propofol effect-site concentration trajectories from the cleaned induction medication dataset and generates patient-level peak exposure metrics for downstream statistical analysis.

## Purpose

The workflow uses timestamped bolus and infusion records from the analysis-ready propofol dataset to simulate effect-site concentration trajectories under validated propofol PK/PD models. The primary model is the Eleveld model, and the main sensitivity analysis uses the Schnider model.

## Main steps

1. Load the cleaned analysis-ready propofol induction dataset
2. Standardize timestamp and case-level variables needed for simulation
3. Define helper functions for dosing weight and simulation inputs
4. Reconstruct effect-site concentration trajectories using the Eleveld model
5. Compute the primary exposure metric, peak modeled effect-site concentration (`C_{e,max}`)
6. Merge Eleveld-based peak exposure estimates back into the patient-level analytic dataset
7. Repeat the simulation workflow using the Schnider model for sensitivity analysis
8. Export model-specific analytic datasets for downstream regression and figure generation

## Main outputs

- patient-level dataset with Eleveld-based peak effect-site concentration estimates
- patient-level dataset with Schnider-based peak effect-site concentration estimates

## Important notes

- The Eleveld model serves as the primary pharmacokinetic/pharmacodynamic framework used in the manuscript.
- The Schnider model is used as a model-specification sensitivity analysis.
- Sensitivity analyses using alternative cohort definitions or inclusion criteria can be performed by changing the upstream input dataset or filtering parameters rather than duplicating the simulation code.
- Output filenames should be standardized in the public version of this notebook to reflect model type and analysis version.

## Reproducibility note

This notebook documents the simulation workflow used to generate modeled propofol exposure metrics from retrospective medication records. It assumes access to the cleaned analysis-ready dataset generated in the extraction and preprocessing workflow.

In [ ]:
# Install required package in Colab environment
pip install PyTCI


In [ ]:
import pandas as pd
import numpy as np
from PyTCI.models import propofol


## Load and preprocess simulation input dataset

This section loads the analysis-ready propofol induction dataset, standardizes time variables, recovers missing BMI values when possible, and applies the sequential exclusions used before pharmacokinetic simulation.

In [ ]:


# Load the dataset
df = pd.read_csv('propofol_analysis_ready.csv')

# Convert action times to datetime objects for event-based simulation
df['ACTION_TIME'] = pd.to_datetime(df['ACTION_TIME'])

# --- RECOVER BMI ---
# Calculate BMI for patients who have Height and Weight recorded but a missing BMI label
# This is done before attrition to "save" as many patients as possible
bmi_recovery_mask = df['BMI'].isna() & df['WEIGHT_KG'].notna() & df['HEIGHT_IN'].notna()
df.loc[bmi_recovery_mask, 'BMI'] = df['WEIGHT_KG'] / ((df['HEIGHT_IN'] * 0.0254) ** 2)

def report_stats(df_current, step_name):
    print(f"--- {step_name} ---")
    print(f"  Rows remaining: {len(df_current)}")
    print(f"  Patients remaining: {df_current['OR_CASE_ID'].nunique()}")
    print("-" * 30)

report_stats(df, "Initial Data")

# 1. DROP PATIENTS WITH MISSING CRITICAL DATA (Sequential Attrition)
# ---------------------------------------------------------------------------
print("--- Step 1: Sequential Attrition Breakdown ---")

# A. Handle Missing SIG (Dose Signal) first
pats_missing_sig = df[df['SIG'].isna()]['OR_CASE_ID'].unique()
df = df[~df['OR_CASE_ID'].isin(pats_missing_sig)].copy()
print(f" 1. Removed for missing SIG     : {len(pats_missing_sig)} unique patients")

# B. Handle other critical variables from the remaining pool
other_vars = ['AGE', 'WEIGHT_KG', 'HEIGHT_IN', 'SEX', 'ACTION_TIME', 'UNITS', 'BMI']
pats_missing_others = df[df[other_vars].isna().any(axis=1)]['OR_CASE_ID'].unique()
df = df[~df['OR_CASE_ID'].isin(pats_missing_others)].copy()
print(f" 2. Removed for other variables : {len(pats_missing_others)} unique patients")
print("-" * 30)

# 2. SEX FILTERING (Strictly M/F)
initial_count = df['OR_CASE_ID'].nunique()
df = df[df['SEX'].str.lower().isin(['m', 'f'])].copy()
print(f"Removed {initial_count - df['OR_CASE_ID'].nunique()} patients with invalid SEX.")

# 3. PHYSIOLOGICAL BOUNDARY FILTERS
initial_count = df['OR_CASE_ID'].nunique()
df = df[(df['HEIGHT_IN'] >= 40) & (df['HEIGHT_IN'] <= 90)].copy()
df = df[(df['BMI'] >= 8) & (df['BMI'] <= 100)].copy()
print(f"Removed {initial_count - df['OR_CASE_ID'].nunique()} patients due to Height/BMI outliers.")

# *** FIX 2: CREATE HEIGHT_CM ***

df['HEIGHT_CM'] = df['HEIGHT_IN'] * 2.54

# 4. DOSE INTEGRITY & RECALCULATION
df['TOTAL_INFUSION_MG'] = df['TOTAL_INFUSION_MG'].fillna(0)
df['TOTAL_PROP_MG'] = df['TOTAL_BOLUS_MG'] + df['TOTAL_INFUSION_MG']

initial_count = df['OR_CASE_ID'].nunique()
df = df[(df['TOTAL_PROP_MG'] >= 20) & (df['TOTAL_PROP_MG'] <= 700)].copy()
print(f"Removed {initial_count - df['OR_CASE_ID'].nunique()} patients with Total Dose <20 or >700 mg.")

# 5. ANTHROPOMETRIC CALCULATIONS (Devine Formula for IBW)
def calculate_ibw(row):
    height_in = row['HEIGHT_IN']
    sex = str(row['SEX']).lower()
    base_weight = 50.0 if sex == 'm' else 45.5
    over_60 = height_in - 60
    return base_weight + (2.3 * over_60)

df['IBW_DEVINE'] = df.apply(calculate_ibw, axis=1)

df['ABW_KG'] = np.where(
    df['WEIGHT_KG'] > (1.2 * df['IBW_DEVINE']),
    df['IBW_DEVINE'] + 0.4 * (df['WEIGHT_KG'] - df['IBW_DEVINE']),
    df['WEIGHT_KG']
)

# 6. CALCULATE DOSE PER KG (ABW)
df['DOSE_PER_ABW'] = df['TOTAL_PROP_MG'] / df['ABW_KG']

report_stats(df, "Final Cleaned Data")

## Eleveld simulation

In [ ]:

# --- 2. The Simulation Function ---
def run_hybrid_simulation(df, num_examples=10):
    summary_results = []
    example_time_series = []
    total_simulation_time = 900

    unique_ids = df['OR_CASE_ID'].unique()
    example_ids = unique_ids[:num_examples]
    total = len(unique_ids)

    # Optimization: Group the dataframe once before the loop
    grouped = df.groupby('OR_CASE_ID')

    for i, (patient_id, patient_data) in enumerate(grouped):
        if i % 500 == 0: print(f"Processing patient {i} of {total}...")

        patient_info = patient_data.iloc[0]

        # Initialize Eleveld Model
        try:
            model = propofol.Eleveld(
                age=patient_info['AGE'],
                weight=patient_info['WEIGHT_KG'],
                height=patient_info['HEIGHT_CM'],
                sex=patient_info['SEX'].lower()
            )
        except: continue



        # Event Setup
        first_dose_time = patient_data['ACTION_TIME'].min()
        patient_data = patient_data.sort_values('ACTION_TIME')
        patient_data['Rel_Time'] = (patient_data['ACTION_TIME'] - first_dose_time).dt.total_seconds().astype(int)

        events = patient_data.to_dict('records')
        total_effect_site_concentration = np.zeros(total_simulation_time)
        current_time = 0
        active_infusion_rate = 0

        # Simulation Loop
        for j, event in enumerate(events):
            event_time = int(event['Rel_Time'])
            if event_time >= total_simulation_time: break

            # Simulate the gap between the last action and this new event
            while current_time < event_time and current_time < total_simulation_time:
                if active_infusion_rate > 0:
                    model.give_drug(active_infusion_rate)
                model.wait_time(1)
                total_effect_site_concentration[current_time] = model.xeo
                current_time += 1

            # Process the actual event (Bolus vs Infusion)
            units = event.get('UNITS', None)
            sig = event.get('SIG', 0)

            if units == 'MG':
                model.give_drug(sig)
            elif units == 'MCG/KG/MIN':
                active_infusion_rate = (sig * patient_info['WEIGHT_KG']) / 1000 / 60

        # Finish simulation up to 900 seconds after all events are processed
        while current_time < total_simulation_time:
            if active_infusion_rate > 0:
                model.give_drug(active_infusion_rate)
            model.wait_time(1)
            total_effect_site_concentration[current_time] = model.xeo
            current_time += 1

        # --- DATA STORAGE ---
        # 1. Store the Summary (Max) for EVERYONE
        max_c = np.max(total_effect_site_concentration)
        summary_results.append({
            "OR_CASE_ID": patient_id,
            "Max_Effect_Concentration": max_c
        })

        # 2. Store full Time Series ONLY for the examples
        if patient_id in example_ids:
            for t in range(total_simulation_time):
                example_time_series.append({
                    "OR_CASE_ID": patient_id,
                    "Time_Seconds": t,
                    "Concentration": total_effect_site_concentration[t]
                })

    return pd.DataFrame(summary_results), pd.DataFrame(example_time_series)

# --- 3. Execution ---
df_max_all, df_full_subset = run_hybrid_simulation(df, num_examples=5)

# Save results
df_max_all.to_csv('all_patients_max_conc.csv', index=False)
df_full_subset.to_csv('example_patients_full_series.csv', index=False)

In [ ]:
# Prepare Eleveld peak concentration output for merge back into the case-level dataset
max_conc_adj = df_max_all

df_analysis_eleveld = pd.merge(
    df.drop_duplicates(subset='OR_CASE_ID'),
    max_conc_adj,
    on='OR_CASE_ID',
    how='inner'
)

print(df_analysis_eleveld[['OR_CASE_ID', 'AGE', 'WEIGHT_KG', 'Max_Effect_Concentration']].head())

df_analysis_eleveld.to_pickle('df_analysis_eleveld.pkl')

## Schnider simulation

In [ ]:


# ==========================================
# 1. HELPER: ORIGINAL K21 CALCULATION
# ==========================================
def calculate_custom_k21(age):
    """
    Custom k21 parameterization used in the Schnider-based sensitivity workflow.
    Calculates k21 as Cl2/V2 and converts the result to per-minute units.
    """
    v2 = 18.9 - 0.391 * (age - 53)       # Volume of peripheral compartment 1 (L)
    cl2 = (1.29 - 0.024 * (age - 53))    # Clearance CL2 in L/h
    k21_per_hour = cl2 / v2
    return k21_per_hour / 60            # Convert to per minute


# ==========================================
# 2. SCHNIDER SIMULATION
# ==========================================
def run_schnider_simulation(df):
    plot_data = []
    total_simulation_time = 900

    unique_ids = df['OR_CASE_ID'].unique()
    total = len(unique_ids)
    print(f"Starting Schnider Simulation for {total} patients...")

    for idx, patient_id in enumerate(unique_ids):
        if idx % 500 == 0: print(f"Processing patient {idx}/{total}...")

        patient_data = df[df['OR_CASE_ID'] == patient_id].sort_values('ACTION_TIME')
        patient_info = patient_data.iloc[0]

        # 1. SETUP PARAMETERS
        # PK uses ABW (AdjBW); Infusions use Total Weight (WEIGHT_KG)
        model_weight = patient_info['ABW_KG']
        first_dose_time = patient_data['ACTION_TIME'].min()

        # 2. INITIALIZE MODEL & INJECT K21
        model = propofol.Schnider(
            age=patient_info['AGE'],
            weight=model_weight,
            height=patient_info['HEIGHT_CM'],
            sex=patient_info['SEX'].lower()
        )
        model.k21 = calculate_custom_k21(patient_info['AGE'])

        # 3. PREPARE EVENTS (Relative Time)
        patient_data['Rel_Time'] = (patient_data['ACTION_TIME'] - first_dose_time).dt.total_seconds().astype(int)


        stop_event = pd.DataFrame({
            'Rel_Time': [total_simulation_time],
            'UNITS': [None],
            'SIG': [0]
        })
        events_df = pd.concat([patient_data[['Rel_Time', 'UNITS', 'SIG']], stop_event], ignore_index=True)
        events_df = events_df.sort_values('Rel_Time')
        events = events_df.to_dict('records')

        # 4. SIMULATION LOOP
        total_effect_site_concentration = np.zeros(total_simulation_time)
        current_time = 0
        active_infusion_rate = 0

        for i, event in enumerate(events[:-1]):
            event_time = event['Rel_Time']
            if event_time >= total_simulation_time:
                break

            # Calculate duration until the next event
            next_event_time = events[i+1]['Rel_Time']
            duration = next_event_time - event_time
            if duration < 0: duration = 0

            # Step A: Handle any gap before this event (if any)
            if event_time > current_time:
                gap = event_time - current_time
                for _ in range(gap):
                    if current_time >= total_simulation_time: break
                    if active_infusion_rate > 0:
                        model.give_drug(active_infusion_rate)
                    model.wait_time(1)
                    total_effect_site_concentration[current_time] = model.xeo
                    current_time += 1

            # Step B: Process the Event
            units = str(event.get('UNITS', 'None'))
            sig = event.get('SIG', 0)

            if units == 'MG':  # Bolus
                model.give_drug(sig)
                # After bolus, continue existing infusion for 'duration'
                for _ in range(duration):
                    if current_time >= total_simulation_time: break
                    if active_infusion_rate > 0:
                        model.give_drug(active_infusion_rate)
                    model.wait_time(1)
                    total_effect_site_concentration[current_time] = model.xeo
                    current_time += 1

            elif 'MCG/KG/MIN' in units:  # Infusion change
                active_infusion_rate = (sig * patient_info['WEIGHT_KG']) / 1000 / 60
                # Infuse at this rate until next event
                for _ in range(duration):
                    if current_time >= total_simulation_time: break
                    model.give_drug(active_infusion_rate)
                    model.wait_time(1)
                    total_effect_site_concentration[current_time] = model.xeo
                    current_time += 1
            else:
                # No change/stop event: continue current rate until next event
                for _ in range(duration):
                    if current_time >= total_simulation_time: break
                    if active_infusion_rate > 0:
                        model.give_drug(active_infusion_rate)
                    model.wait_time(1)
                    total_effect_site_concentration[current_time] = model.xeo
                    current_time += 1

        # 5. FINAL DATA STORAGE
        max_c = np.max(total_effect_site_concentration)
        plot_data.append({
            "OR_CASE_ID": patient_id,
            "Max_Effect_Concentration": max_c
        })

    return pd.DataFrame(plot_data)

# --- EXECUTION ---
df_max_schnider = run_schnider_simulation(df)
df_max_schnider.to_csv('all_patients_max_conc_schnider.csv', index=False)

In [ ]:
# Prepare Schnider peak concentration output for merge back into the case-level dataset
df_analysis_schnider = pd.merge(
    df.drop_duplicates(subset='OR_CASE_ID'),
    df_max_schnider,
    on='OR_CASE_ID',
    how='inner'
)

print(df_analysis_schnider[['OR_CASE_ID', 'AGE', 'WEIGHT_KG', 'Max_Effect_Concentration']].head())

df_analysis_schnider.to_pickle('df_analysis_schnider.pkl')